In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import re
from collections import Counter

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [ ]:
!curl -SOL https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [2]:
def tokenize(s):
    s = s.lower()
    s = re.sub(r"\s+", " ", s)
    tokens = re.findall(r"[a-z]+|[0-9]+|[^\w\s]", s)
    return tokens

In [3]:
with open("input.txt") as f:
    dataset = f.read()

dataset = tokenize(dataset)
print(f"Input tokens: {len(dataset)}")

MAX_VOCAB = 4000
vocab = list(set(dataset))
counts = Counter(dataset)
vocab.sort(key=lambda x: counts[x], reverse=True)
vocab = vocab[:min(MAX_VOCAB - 4, len(vocab))]
# Force eval 1/2 words to exist in vocab
n_force = 0
for i, w in enumerate(["love", "death", "man", "woman", "i", "polish", "the", "mirror", "nail", "is", "red"]):
    if w in vocab:
        continue
    j = -(n_force + 1)
    print(f"Forcing {w} into vocab (replacing {vocab[j]} [{counts[vocab[j]]} instances])")
    vocab[j] = w
    n_force += 1
del counts, n_force

vocab += ["<PAD>", "<UNK>", "<BOS>", "<EOS>"]
vocab_size = len(vocab)
print(f"Vocab size: {vocab_size}")

token_to_id = {t: i for i, t in enumerate(vocab)}
id_to_token = {i: t for t, i in token_to_id.items()}
ids = [token_to_id.get(w, token_to_id["<UNK>"]) for w in dataset]
split = int(0.9 * len(ids))
train_ids = torch.tensor(ids[:split], dtype=torch.long)
test_ids  = torch.tensor(ids[split:], dtype=torch.long)

Input tokens: 262927
Forcing polish into vocab (replacing crook [3 instances])
Forcing mirror into vocab (replacing dissentious [3 instances])
Vocab size: 4000


In [4]:
SEQ_LEN = 32
BATCH_SIZE = 128

def get_batch(data_ids, batch_size, seq_len):
    max_start = len(data_ids) - (seq_len + 1)
    starts = torch.randint(0, max_start, (batch_size,))
    x = torch.stack([
        data_ids[s:s + seq_len]
        for s in starts
    ])
    y = torch.stack([
        data_ids[s + 1:s + 1 + seq_len]
        for s in starts
    ])
    return x.to(device), y.to(device)

def get_bilm_batch(data_ids, batch_size, seq_len):
    L = len(data_ids)
    max_start = L - (seq_len + 1)
    assert max_start > 0, "Data too small for the chosen seq_len."

    starts = torch.randint(0, max_start, (batch_size,))
    x = torch.stack([data_ids[int(s.item()): int(s.item()) + seq_len] for s in starts])
    y_next = torch.stack([data_ids[int(s.item()) + 1: int(s.item()) + 1 + seq_len] for s in starts])

    x_rev = torch.flip(x, dims=[1])
    y_next_rev = torch.flip(y_next, dims=[1])

    return x.to(device), y_next.to(device), x_rev.to(device), y_next_rev.to(device)

def make_reversal_batch(data_ids, batch_size, seq_len):
    max_start = len(data_ids) - seq_len
    starts = torch.randint(0, max_start, (batch_size,))
    src = torch.stack([data_ids[s:s+seq_len] for s in starts])
    tgt = torch.flip(src, dims=[1])
    bos = torch.full((batch_size, 1), token_to_id["<BOS>"], dtype=torch.long)
    eos = torch.full((batch_size, 1), token_to_id["<EOS>"], dtype=torch.long)
    tgt_in  = torch.cat([bos, tgt], dim=1)
    tgt_out = torch.cat([tgt, eos], dim=1)
    return src.to(device), tgt_in.to(device), tgt_out.to(device)

In [5]:
class LSTM(nn.Module):
    def __init__(self, vocab_size, embed_size=128, hidden_size=256):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, x):
        embed = self.embedding(x)
        h, _ = self.lstm(embed)
        return self.output(h), h
    
    def do_train(self, num_iters, alpha, batch_size=BATCH_SIZE, seq_len=SEQ_LEN):
        opt = torch.optim.Adam(self.parameters(), lr=alpha)
        self.train()
        print("Training", end="")
        for i in range(num_iters):
            x, y = get_batch(train_ids, batch_size, seq_len)
            logits, _ = self(x)
            loss = F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1))
            
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(self.parameters(), 1.0)
            opt.step()
            
            if i % (num_iters // 100) == 0:
                print(".", end="")
        self.eval()
        print("Done")

class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_size=128, hidden_size=256):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm_f = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.lstm_r = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.output_f = nn.Linear(hidden_size, vocab_size)
        self.output_r = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, x):
        embed = self.embedding(x)
        h_f, _ = self.lstm_f(embed)
        out_f = self.output_f(h_f)

        embed_r = torch.flip(embed, dims=[1])
        h_r, _ = self.lstm_r(embed_r)
        out_r = self.output_r(h_r)

        return out_f, out_r, torch.cat([h_f, h_r], dim=-1)
    
    def do_train(self, num_iters, alpha, batch_size=BATCH_SIZE, seq_len=SEQ_LEN):
        opt = torch.optim.Adam(self.parameters(), lr=alpha)
        self.train()
        print("Training", end="")
        for i in range(num_iters):
            x, y_next, x_rev, y_next_rev = get_bilm_batch(train_ids, batch_size, seq_len)
            logits_fwd, logits_bwd, _ = self(x)
        
            loss_f = F.cross_entropy(logits_fwd.reshape(-1, vocab_size), y_next.reshape(-1))
            loss_b = F.cross_entropy(logits_bwd.reshape(-1, vocab_size), y_next_rev.reshape(-1))
            loss = loss_f + loss_b
        
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(self.parameters(), 1.0)
            opt.step()
            if i % (num_iters // 100) == 0:
                print(".", end="")
        self.eval()
        print("Done")

class Seq2Seq(nn.Module):
    def __init__(self, vocab_size, embed_size=128, hidden_size=256):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=token_to_id["<PAD>"])
        self.encoder = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.decoder = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, vocab_size)
    
    def encode(self, x):
        embed = self.embedding(x)
        _, (h, c) = self.encoder(embed)
        return (h, c)
    
    def decode(self, tgt, state):
        embed = self.embedding(tgt)
        out, state = self.decoder(embed, state)
        return self.output(out), state
    
    def forward(self, x, tgt):
        out, _ = self.decode(tgt, self.encode(x))
        return out
    
    def do_train(self, num_iters, alpha, batch_size=BATCH_SIZE, seq_len=SEQ_LEN):
        opt = torch.optim.Adam(self.parameters(), lr=alpha)
        self.train()
        print("Training", end="")
        for i in range(num_iters):
            src, tgt_in, tgt_out = make_reversal_batch(train_ids, batch_size, seq_len)
            logits = self(src, tgt_in)
            loss = F.cross_entropy(logits.reshape(-1, vocab_size), tgt_out.reshape(-1))
            
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(self.parameters(), 1.0)
            opt.step()
            
            if i % (num_iters // 100) == 0:
                print(".", end="")
        self.eval()
        print("Done")

In [11]:
model1 = LSTM(vocab_size).to(device)
model1.do_train(10000, 3e-3)

model2 = BiLSTM(vocab_size).to(device)
model2.do_train(10000, 3e-3)

model3 = Seq2Seq(vocab_size).to(device)
model3.do_train(10000, 3e-3)

Training....................................................................................................Done
Training....................................................................................................Done
Training....................................................................................................Done


# Eval 1

In [12]:
def neighbors(embeddings, word, n=8):
    assert word in token_to_id
    id_word = token_to_id[word]
    e_word = embeddings[id_word]
    sims = F.cosine_similarity(e_word.unsqueeze(0), embeddings, dim=1)
    vals, idxs = torch.topk(sims, k=min(n + 1, embeddings.shape[0]))
    return [
        (id_to_token[i], v)
        for v, i in zip(vals.tolist(), idxs.tolist())
        if i != id_word
    ]

for model, embds in [("LSTM", model1.embedding.weight), ("BiLSTM", model2.embedding.weight), ("Seq2Seq", model3.embedding.weight)]:
    print(f"{model}:")
    for word in ["love", "death", "man", "woman"]:
        print(f"    {word}")
        for w, v in neighbors(embds, word):
            print(f"        {w:<15}({v:.5f})")
    print()

LSTM:
    love
        joy            (0.30320)
        woe            (0.29042)
        razed          (0.27543)
        sounded        (0.27305)
        cur            (0.26952)
        bids           (0.26255)
        sell           (0.26048)
        lead           (0.25778)
    death
        poison         (0.33648)
        madman         (0.30473)
        inform         (0.29376)
        fellow         (0.29372)
        lap            (0.29215)
        bell           (0.28845)
        christ         (0.28629)
        sat            (0.27672)
    man
        firm           (0.36965)
        direction      (0.32672)
        murdering      (0.29343)
        sir            (0.28771)
        language       (0.28458)
        thereto        (0.28257)
        built          (0.27931)
        gloucester     (0.26589)
    woman
        corse          (0.30895)
        evils          (0.27653)
        fool           (0.27224)
        followers      (0.26742)
        courtier       (0.26710)


# Eval 2

In [13]:
def embed_sequence(model_func, seq):
    tokens = tokenize(seq)
    ids = [token_to_id.get(t, token_to_id["<UNK>"]) for t in tokens]
    ids.append(token_to_id["<EOS>"])
    return model_func(torch.tensor(ids).to(device))

# We want the hidden state for each token, but `model3.encode` returns only the final state
def m3_eval(model, x):
    e = model.embedding(x)
    o, _ = model.encoder(e)
    return o

for model, func in [("LSTM", lambda x: model1.forward(x)[-1]), ("BiLSTM", lambda x: model2.forward(x)[-1]), ("Seq2Seq", lambda x: m3_eval(model3, x))]:
    s1 = embed_sequence(func, "i polish the mirror.")
    s2 = embed_sequence(func, "the nail polish is red.")
    polish1 = s1.tolist()[1]
    polish2 = s2.tolist()[2]
    sim = np.dot(polish1, polish2) / (np.linalg.norm(polish1) * np.linalg.norm(polish2))
    print(f"{model}: {sim:.5f}")

LSTM: 0.61478
BiLSTM: 0.63836
Seq2Seq: 0.60177


# Insight Questions

## Q1

- LSTM:
    - static: `self.embedding.weight`
    - contextual: `h` (first return value of `self.lstm`)
- BiLSTM:
    - static: `self.embedding.weight`
    - contextual: concat of `h_f` and `h_r` (first return values of `self.lstm_f` and `self.lstm_r`)
- Seq2Seq:
    - static: `self.embedding.weight`
    - contextual: first return value of `self.encoder`

## Q2

The cosine similarity of `polish` from the LSTM to the BiLSTM barely increased ($\approx+0.02$). This is because the short length of the sequences means that the BiLSTM's main advantage of pulling info from both sides does not matter that much.

## Q3

The fundemental difference between the Seq2Seq models and the other two is that the Seq2Seq model is conditional (the others are unconditional). The Seq2Seq generates its output conditioned on the result of the encoding process. In comparison, the output of other two models only depends on the input sequence. The Seq2Seq model also can produce variable length outputs, while the other models can only produce outputs from the final hidden state.

## Q4

Selected change:
> reduce sequence length by half

In [14]:
new_seq_len = SEQ_LEN // 2

q4_model1 = LSTM(vocab_size).to(device)
q4_model1.do_train(10000, 3e-3, seq_len=new_seq_len)
q4_model2 = BiLSTM(vocab_size).to(device)
q4_model2.do_train(10000, 3e-3, seq_len=new_seq_len)
q4_model3 = Seq2Seq(vocab_size).to(device)
q4_model3.do_train(10000, 3e-3, seq_len=new_seq_len)

for model, func in [("LSTM", lambda x: q4_model1.forward(x)[-1]), ("BiLSTM", lambda x: q4_model2.forward(x)[-1]), ("Seq2Seq", lambda x: m3_eval(q4_model3, x))]:
    s1 = embed_sequence(func, "i polish the mirror.")
    s2 = embed_sequence(func, "the nail polish is red.")
    polish1 = s1.tolist()[1]
    polish2 = s2.tolist()[2]
    sim = np.dot(polish1, polish2) / (np.linalg.norm(polish1) * np.linalg.norm(polish2))
    print(f"{model}: {sim:.5f}")

Training....................................................................................................Done
Training....................................................................................................Done
Training....................................................................................................Done
LSTM: 0.48937
BiLSTM: 0.62886
Seq2Seq: 0.77330


Result:
- LSTM's value decreased ($\approx-0.13$)
- BiLSTM's value decreased slightly ($\approx-0.01$)
- Seq2Seq's value increased ($\approx+0.17$)

Why:
- Reduced sequence length means the models learned less contextual information during training